# Cartographie prédictive GWS — KNeighborsRegressor (KNN)

Notebook Google Colab généré à partir du script Python d'origine : **`KNN-Regressor.py`**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Chaque cellule exécute directement son étape, comme le script d'origine (qui n'a pas de fonction `main()`).
Les résultats sont écrits dans `input_data_ML/GWS_PREDICTIONS_GEE_KNN` sur Drive.

Le code du script est repris tel quel. Les seules lignes ajoutées ou modifiées pour Colab
sont repérées par le commentaire `# [COLAB]`.

### Description (en-tête du script d'origine)

```text
# ============================================================
#  CARTOGRAPHIE PREDICTIVE GWS (KNN)
#  (TRAIN = longue période, TEST = 5% des dates, CLIP GEE, SANS SOUS-ÉCHANTILLONNAGE)
#
#  ✅ Version ROBUSTE :
#   - Résout automatiquement les noms de sections dans name_files.txt
#     (insensible à la casse/espaces/_/- + alias Precipitations_IDW, Precipitations_API_IDW, etc.)
#   - Extrait dates depuis noms en gérant YYYYMMDD et DDMMYYYY
#
#  - Modèle : KNeighborsRegressor sur grille GLDAS (~25 km)
#  - Features : [CHIRPS, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, sinDOY, cosDOY]
#  - Rééchantillonnage NoData correct + remplissage NaN (cartes SANS TROUS dans Regions)
#  - Clip de toutes les cartes (test + forecast) sur la zone Regions reconstruite
#
#  ➤ Périodes utilisées :
#    TRAIN pool : 2021-01-01 → 2025-06-30
#    TEST = 5% des dates tirées aléatoirement dans cette fenêtre
#    FORECAST : 2025-07-01 → 2025-10-30
#
#  NB : les dates réellement utilisées dépendent de l’intersection :
#       CHIRPS ∩ LST ∩ GWS ∩ Qsb ∩ RZSM ∩ SoilM.
# ============================================================
```

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab (scikit-learn, numpy et joblib le sont).

In [ ]:
!pip install -q rasterio

### A.3 Imports

In [ ]:
import os
import re
import time
import bisect
from datetime import datetime, date

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import joblib

# [COLAB] Versions des bibliothèques utilisées
from importlib.metadata import version as _pkg_version
for _pkg in ('numpy', 'scikit-learn', 'rasterio'):
    print(f"{_pkg:13s}: {_pkg_version(_pkg)}")

## 1. PARAMÈTRES GÉNÉRAUX

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# ------------------------------------------------------------
# 1) PARAMÈTRES GÉNÉRAUX
# ------------------------------------------------------------
np.random.seed(42)

# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR  = "/content/drive/MyDrive/input_data_ML"
NAME_FILE = os.path.join(BASE_DIR, "name_files.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon le dossier de sortie serait créé hors de Drive et perdu en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

# Sous-dossiers (debug / affichage seulement)
CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_Clipped")
LST_DIR    = os.path.join(BASE_DIR, "MODIS_LST_Clipped")
NDVI_DIR   = os.path.join(BASE_DIR, "MODIS_NDVI_Clipped")
ET_DIR     = os.path.join(BASE_DIR, "MODIS_ET_Clipped")
GWS_DIR    = os.path.join(BASE_DIR, "GLDAS_GWS_Clipped")
QSB_DIR    = os.path.join(BASE_DIR, "ERA5_Qsb_Clipped")
RZSM_DIR   = os.path.join(BASE_DIR, "SMAP_RZSM_Clipped")
SOILM_DIR  = os.path.join(BASE_DIR, "SMAP_SoilM_Clipped")

OUT_DIR = os.path.join(BASE_DIR, "GWS_PREDICTIONS_GEE_KNN")
os.makedirs(OUT_DIR, exist_ok=True)

NODATA_VALUE  = -9999.0
TEST_FRACTION = 0.05  # ✅ 5% des dates (tu avais 0.1 dans ton script)

# Fenêtres temporelles
TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

FORECAST_START = datetime(2025, 7, 1)
FORECAST_END   = datetime(2025, 10, 30)

print("=== PARAMÈTRES GÉNÉRAUX ===")
print(" BASE_DIR :", BASE_DIR)
print(" NAME_FILE :", NAME_FILE)
print(" OUT_DIR :", OUT_DIR)
print(" TRAIN_START/END :", TRAIN_START, "→", TRAIN_END)
print(" TEST_FRACTION :", TEST_FRACTION)
print(" FORECAST_START/END :", FORECAST_START.date(), "→", FORECAST_END.date())
print("============================\n")

# Dates exclues (LST manquante 2022-10-11 → 2022-10-22)
EXCLUDED_DATES = set(date(2022, 10, d) for d in range(11, 23))
print("Dates exclues (LST manquante) :")
for d in sorted(EXCLUDED_DATES):
    print(" -", d)

## 2. UTILITAIRES : parse name_files + résolution de sections

Adaptation Colab : les entrées de `name_files.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# ------------------------------------------------------------
# 2) UTILITAIRES : parse name_files + résolution de sections
# ------------------------------------------------------------
def normalize_section_name(s: str) -> str:
    return re.sub(r"[\s_\-]+", "", s.strip().lower())

def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p


def parse_name_file(txt_path: str) -> dict:
    sections = {}
    current = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
                continue
            if current is None:
                continue
            sections[current].append(_win_to_colab_path(line))  # [COLAB] chemin Windows -> Colab
    return sections

def resolve_section_name(sections: dict, wanted: str, aliases=None) -> str:
    if aliases is None:
        aliases = []

    # 1) exact
    if wanted in sections:
        return wanted

    # 2) match normalisé
    wn = normalize_section_name(wanted)
    for k in sections.keys():
        if normalize_section_name(k) == wn:
            return k

    # 3) alias exact + normalisé
    for a in aliases:
        if a in sections:
            return a
        an = normalize_section_name(a)
        for k in sections.keys():
            if normalize_section_name(k) == an:
                return k

    # 4) fallback "contient le mot clé"
    key = wanted.split("_")[0].lower()
    for k in sections.keys():
        if key in k.lower():
            return k

    print("\n❌ Section introuvable :", wanted)
    print("✅ Sections disponibles dans name_files.txt :")
    for s in sorted(sections.keys()):
        print(" -", s)
    raise RuntimeError(f"Section introuvable pour '{wanted}'. Essayé alias: {aliases}")

def extract_date_from_filename(path: str) -> date:
    """
    Extrait une date depuis le nom du fichier.
    - essaie YYYYMMDD puis DDMMYYYY.
    """
    name = os.path.basename(path)
    m = re.search(r"(\d{8})", name)
    if not m:
        raise ValueError(f"Impossible d'extraire une date (8 chiffres) depuis: {name}")
    s = m.group(1)

    try:
        return datetime.strptime(s, "%Y%m%d").date()
    except ValueError:
        pass
    try:
        return datetime.strptime(s, "%d%m%Y").date()
    except ValueError:
        raise ValueError(f"Format date inconnu pour {name} (token={s})")

def build_date_to_path_dict_from_section(sections: dict, wanted_section: str, aliases=None) -> dict:
    real = resolve_section_name(sections, wanted_section, aliases=aliases)

    if real != wanted_section:
        print(f"ℹ️ Section '{wanted_section}' résolue → '{real}'")

    d = {}
    for rel_path in sections[real]:
        abs_path = os.path.normpath(os.path.join(BASE_DIR, rel_path))
        if not os.path.isfile(abs_path):
            print(f"⚠️ Fichier introuvable (ignoré) : {abs_path}")
            continue
        try:
            dk = extract_date_from_filename(abs_path)
            d[dk] = abs_path
        except ValueError as e:
            print("⚠️", e)
    return d

def build_prev_date_support(date_dict: dict):
    """Retourne get_prev(target_date): dernière date dispo <= target_date."""
    dates_sorted = sorted(date_dict.keys())
    def get_prev(target_date):
        if not dates_sorted:
            return None
        idx = bisect.bisect_right(dates_sorted, target_date) - 1
        if idx < 0:
            return None
        return dates_sorted[idx]
    return get_prev

## 3. UTILITAIRES RASTERS

In [ ]:
# ------------------------------------------------------------
# 3) UTILITAIRES RASTERS
# ------------------------------------------------------------
def read_raster_raw(path: str):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
    return arr, profile, nodata

def to_float_with_nan(arr: np.ndarray, nodata):
    arr = arr.astype("float32")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr

def reproject_to_template(
    src_arr, src_profile, template_profile,
    src_nodata=None, dst_nodata=np.nan,
    resampling=Resampling.average
):
    dst_arr = np.full(
        (template_profile["height"], template_profile["width"]),
        dst_nodata,
        dtype="float32"
    )
    kwargs = dict(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        dst_transform=template_profile["transform"],
        dst_crs=template_profile["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    if src_nodata is not None:
        kwargs["src_nodata"] = src_nodata
    reproject(**kwargs)
    return dst_arr

def fill_nan_with_mean(arr: np.ndarray):
    if np.all(np.isnan(arr)):
        return arr
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr)

## 4. MASQUE REGIONS (reconstruit depuis rasters)

In [ ]:
# ------------------------------------------------------------
# 4) MASQUE REGIONS (reconstruit depuis rasters)
# ------------------------------------------------------------
REGION_MASK = None

def update_region_mask(precip_res, lst_res, ndvi_res, template_profile):
    global REGION_MASK
    base_mask = ~(np.isnan(precip_res) & np.isnan(lst_res) & np.isnan(ndvi_res))
    if REGION_MASK is None:
        REGION_MASK = base_mask
        print("\n🗺️ Masque 'Regions' initial construit à partir des rasters (1er jour).")
    else:
        REGION_MASK = REGION_MASK | base_mask

def get_region_mask(template_profile):
    global REGION_MASK
    if REGION_MASK is None:
        print("⚠️ REGION_MASK non initialisé → on prend toute la grille.")
        REGION_MASK = np.ones((template_profile["height"], template_profile["width"]), dtype=bool)
    return REGION_MASK

### Contrôle des entrées de `name_files.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichier présent sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _missing = [_e for _e in _entries
                if not os.path.isfile(os.path.normpath(os.path.join(BASE_DIR, _e)))]
    _n_ok = len(_entries) - len(_missing)
    _flag = "✅" if (_n_ok > 0 and not _missing) else ("⚠️" if _n_ok > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_ok} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {os.path.normpath(os.path.join(BASE_DIR, _e))}")

## 5. INDEXATION DES FICHIERS (via name_files.txt) - ROBUSTE

In [ ]:
# ------------------------------------------------------------
# 5) INDEXATION DES FICHIERS (via name_files.txt) - ROBUSTE
# ------------------------------------------------------------
print("\n=== INDEXATION DES FICHIERS PAR DATE (name_files.txt) ===")
sections = parse_name_file(NAME_FILE)

chirps_dict = build_date_to_path_dict_from_section(
    sections, "CHIRPS_Clipped",
    aliases=[
        "CHIRPS", "CHIRPS_CLIPPED", "CHIRPS-Clipped",
        "Precipitations_IDW", "Precipitations_API_IDW", "Precipitations_API", "Precipitations"
    ]
)
lst_dict   = build_date_to_path_dict_from_section(sections, "MODIS_LST_Clipped",  aliases=["LST", "MODIS_LST", "MODISLST"])
ndvi_dict  = build_date_to_path_dict_from_section(sections, "MODIS_NDVI_Clipped", aliases=["NDVI", "MODIS_NDVI", "MODISNDVI"])
et_dict    = build_date_to_path_dict_from_section(sections, "MODIS_ET_Clipped",   aliases=["ET", "MODIS_ET", "MODISET"])
gws_dict   = build_date_to_path_dict_from_section(sections, "GLDAS_GWS_Clipped",  aliases=["GWS", "GLDAS_GWS", "GLDASGWS"])
qsb_dict   = build_date_to_path_dict_from_section(sections, "ERA5_Qsb_Clipped",   aliases=["QSB", "ERA5_QSB", "ERA5QSB"])
rzsm_dict  = build_date_to_path_dict_from_section(sections, "SMAP_RZSM_Clipped",  aliases=["RZSM", "SMAP_RZSM", "SMAPRZSM"])
soilm_dict = build_date_to_path_dict_from_section(sections, "SMAP_SoilM_Clipped", aliases=["SOILM", "SMAP_SOILM", "SMAPSoilM", "SMAP_SoilM"])

print("\nNb dates CHIRPS :", len(chirps_dict))
print("Nb dates LST    :", len(lst_dict))
print("Nb dates NDVI   :", len(ndvi_dict))
print("Nb dates ET     :", len(et_dict))
print("Nb dates GWS    :", len(gws_dict))
print("Nb dates Qsb    :", len(qsb_dict))
print("Nb dates RZSM   :", len(rzsm_dict))
print("Nb dates SoilM  :", len(soilm_dict))

get_ndvi_date_for = build_prev_date_support(ndvi_dict)
get_et_date_for   = build_prev_date_support(et_dict)

## 6. DATES TRAIN/TEST (split sur les DATES, pas sur les pixels)

In [ ]:
# ------------------------------------------------------------
# 6) DATES TRAIN/TEST (split sur les DATES, pas sur les pixels)
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DES DATES TRAIN / TEST ===")

all_common_dates = sorted(
    set(chirps_dict.keys())
    & set(lst_dict.keys())
    & set(gws_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

print("Nombre total de dates communes (CHIRPS ∩ LST ∩ GWS ∩ Qsb ∩ RZSM ∩ SoilM) :", len(all_common_dates))
if not all_common_dates:
    raise RuntimeError("⚠️ Aucune date commune entre les variables journalières.")

date_candidates = [
    d for d in all_common_dates
    if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)
]
if not date_candidates:
    raise RuntimeError("⚠️ Aucune date candidate dans la fenêtre TRAIN (après exclusions).")

train_dates, test_dates = train_test_split(
    date_candidates,
    test_size=TEST_FRACTION,
    random_state=42,
    shuffle=True
)
train_dates = sorted(train_dates)
test_dates  = sorted(test_dates)

print(f"\nFenêtre TRAIN (pool) : {TRAIN_START} → {TRAIN_END}")
print(f" → Nombre de dates train : {len(train_dates)} ({min(train_dates)} → {max(train_dates)})")
print(f"\nTEST = {int(TEST_FRACTION*100)}% des dates dans la même fenêtre")
print(f" → Nombre de dates test : {len(test_dates)} ({min(test_dates)} → {max(test_dates)})")

## 7. CONSTRUCTION DATASET (TRAIN + TEST, tous pixels valides)

In [ ]:
# ------------------------------------------------------------
# 7) CONSTRUCTION DATASET (TRAIN + TEST, tous pixels valides)
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DU DATASET (pixels sur toutes les dates) ===")

X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []
template_profile = None

def build_Xy_for_date(date_key: date, template_profile, role="train"):
    # NDVI/ET (dernière date <= date_key)
    ndvi_date = get_ndvi_date_for(date_key)
    if ndvi_date is None:
        print(f"⏭️ [{role}] {date_key} : NDVI indisponible (<= date), ignoré.")
        return None, None, template_profile

    et_date = get_et_date_for(date_key)
    if et_date is None:
        print(f"⏭️ [{role}] {date_key} : ET indisponible (<= date), ignoré.")
        return None, None, template_profile

    # GWS (template)
    if date_key not in gws_dict:
        print(f"⏭️ [{role}] {date_key} : GWS manquant, ignoré.")
        return None, None, template_profile

    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    if template_profile is None:
        template_profile = gws_prof
        print("\nGrille GLDAS (template) :")
        print(" - CRS :", template_profile["crs"])
        print(" - Taille:", template_profile["width"], "x", template_profile["height"])
        print(" - Résolution approx:", template_profile["transform"][0])

    # Variables journalières
    needed = [chirps_dict, lst_dict, qsb_dict, rzsm_dict, soilm_dict]
    if not all(date_key in dct for dct in needed):
        print(f"⏭️ [{role}] {date_key} : variable journalière manquante, ignoré.")
        return None, None, template_profile

    precip_raw, precip_prof, precip_nodata = read_raster_raw(chirps_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata)

    update_region_mask(precip_res, lst_res, ndvi_res, template_profile)
    region_mask = get_region_mask(template_profile)

    # Remplissage NaN (cartes sans trous)
    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    # Pixels valides = GWS défini + dans Regions
    mask_valid = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_valid):
        print(f"⏭️ [{role}] {date_key} : aucun pixel valide, ignoré.")
        return None, None, template_profile

    # DOY sin/cos
    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n = np.count_nonzero(mask_valid)
    X_day = np.stack([
        precip_f[mask_valid],
        lst_f[mask_valid],
        ndvi_f[mask_valid],
        et_f[mask_valid],
        qsb_f[mask_valid],
        rzsm_f[mask_valid],
        soilm_f[mask_valid],
        np.full(n, sin_doy, dtype="float32"),
        np.full(n, cos_doy, dtype="float32"),
    ], axis=1).astype("float32")

    y_day = gws_arr[mask_valid].astype("float32")

    # Sécurité NaN
    row_ok = (~np.isnan(y_day)) & (~np.isnan(X_day).any(axis=1))
    if not np.any(row_ok):
        print(f"⏭️ [{role}] {date_key} : lignes invalides (NaN), ignoré.")
        return None, None, template_profile

    X_day = X_day[row_ok]
    y_day = y_day[row_ok]

    print(f" ✔️ [{role}] {date_key} : {X_day.shape[0]} pixels utilisés.")
    return X_day, y_day, template_profile


# Boucle TRAIN
for d in train_dates:
    Xd, yd, template_profile = build_Xy_for_date(d, template_profile, role="train")
    if Xd is None:
        continue
    X_train_list.append(Xd)
    y_train_list.append(yd)

# Boucle TEST
for d in test_dates:
    Xd, yd, template_profile = build_Xy_for_date(d, template_profile, role="test")
    if Xd is None:
        continue
    X_test_list.append(Xd)
    y_test_list.append(yd)

if not X_train_list:
    raise RuntimeError("⚠️ Aucun pixel d'entraînement valide n'a été trouvé.")
if not X_test_list:
    raise RuntimeError("⚠️ Aucun pixel de test valide n'a été trouvé.")

X_train = np.vstack(X_train_list).astype("float32")
y_train = np.concatenate(y_train_list).astype("float32")
X_test  = np.vstack(X_test_list).astype("float32")
y_test  = np.concatenate(y_test_list).astype("float32")

print(f"\nNombre total de pixels d'entraînement : {X_train.shape[0]}")
print(f"Nombre total de pixels de test       : {X_test.shape[0]}")
print("Exemple X_train[0] :", X_train[0])

## 8. ENTRAÎNEMENT KNN (StandardScaler + KNN)

In [ ]:
# ------------------------------------------------------------
# 8) ENTRAÎNEMENT KNN (StandardScaler + KNN)
# ------------------------------------------------------------
print("\n=== ENTRAÎNEMENT DU MODÈLE KNN ===")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    shuffle=True
)
print(f"Taille X_tr : {X_tr.shape}, X_val : {X_val.shape}")

knn_reg = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(
        n_neighbors=7,
        weights="distance",
        metric="minkowski",
        p=2,
        n_jobs=-1,
    )),
])

print("🔄 Fit KNN ...")
t0 = time.time()
knn_reg.fit(X_tr, y_tr)
t1 = time.time()
training_time_ms = (t1 - t0) * 1000.0

# TRAIN / VAL / TEST (pixels)
y_tr_pred  = knn_reg.predict(X_tr)
y_val_pred = knn_reg.predict(X_val)
y_te_pred  = knn_reg.predict(X_test)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

print("\n╔═════════ MODÈLE KNN ═════════╗")
print(f" ➤ Temps d'entraînement : {training_time_ms:.2f} ms")
print(" --- TRAIN (X_tr) ---")
print(f"    R²   : {r2_score(y_tr, y_tr_pred):.4f}")
print(f"    MAE  : {mean_absolute_error(y_tr, y_tr_pred):.4f}")
print(f"    RMSE : {rmse(y_tr, y_tr_pred):.4f}")
print(" --- VALID (X_val) ---")
print(f"    R²   : {r2_score(y_val, y_val_pred):.4f}")
print(f"    MAE  : {mean_absolute_error(y_val, y_val_pred):.4f}")
print(f"    RMSE : {rmse(y_val, y_val_pred):.4f}")
print(" --- TEST (dates 5% agrégées en pixels) ---")
print(f"    R²   : {r2_score(y_test, y_te_pred):.4f}")
print(f"    MAE  : {mean_absolute_error(y_test, y_te_pred):.4f}")
print(f"    RMSE : {rmse(y_test, y_te_pred):.4f}")
print("╚══════════════════════════════╝")

# Sauvegarde modèle
model_path = os.path.join(OUT_DIR, "KNN_GWS_model.joblib")
joblib.dump(knn_reg, model_path)
print("💾 Modèle sauvegardé:", model_path)

## 9. ÉVALUATION CARTOGRAPHIQUE (dates TEST) + sauvegarde rasters

In [ ]:
# ------------------------------------------------------------
# 9) ÉVALUATION CARTOGRAPHIQUE (dates TEST) + sauvegarde rasters
# ------------------------------------------------------------
print("\n=== ÉVALUATION CARTOGRAPHIQUE (dates TEST) ===")

region_mask = get_region_mask(template_profile)
out_profile_test = template_profile.copy()
out_profile_test.update(dtype="float32", count=1, nodata=NODATA_VALUE)

all_true, all_pred = [], []

for date_key in test_dates:
    print(f"\n🧪 Date TEST: {date_key}")

    if date_key in EXCLUDED_DATES:
        print(" ⏭️ exclue (LST manquante)")
        continue

    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(" ⏭️ NDVI/ET indisponible")
        continue
    if date_key not in gws_dict:
        print(" ⏭️ GWS manquant")
        continue
    if not all(date_key in dct for dct in [chirps_dict, lst_dict, qsb_dict, rzsm_dict, soilm_dict]):
        print(" ⏭️ variable journalière manquante")
        continue

    # vérité terrain
    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    # entrées
    precip_raw, precip_prof, precip_nodata = read_raster_raw(chirps_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pix = precip_f.size
    X_all = np.stack([
        precip_f.ravel(),
        lst_f.ravel(),
        ndvi_f.ravel(),
        et_f.ravel(),
        qsb_f.ravel(),
        rzsm_f.ravel(),
        soilm_f.ravel(),
        np.full(n_pix, sin_doy, dtype="float32"),
        np.full(n_pix, cos_doy, dtype="float32"),
    ], axis=1).astype("float32")

    if np.isnan(X_all).any():
        print(" ⏭️ X_all contient NaN → ignorée")
        continue

    y_pred_all = knn_reg.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    mask_gws = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_gws):
        print(" ⚠️ Aucun pixel valide pour métriques")
        continue

    y_true = gws_arr[mask_gws]
    y_pred = pred_clip[mask_gws]

    r2_d   = r2_score(y_true, y_pred)
    mae_d  = mean_absolute_error(y_true, y_pred)
    rmse_d = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f" ➤ R²  : {r2_d:.4f}")
    print(f" ➤ MAE : {mae_d:.4f}")
    print(f" ➤ RMSE: {rmse_d:.4f}")

    all_true.append(y_true)
    all_pred.append(y_pred)

    out_name = os.path.join(OUT_DIR, f"GWS_TEST_PRED_KNN_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name, "w", **out_profile_test) as dst:
        dst.write(pred_clip, 1)
    print(" ✔️ Raster TEST sauvegardé:", out_name)

if all_true:
    all_true_flat = np.concatenate(all_true)
    all_pred_flat = np.concatenate(all_pred)
    print("\n📊 MÉTRIQUES GLOBALES (TEST dates)")
    print(" ➤ R² global  :", r2_score(all_true_flat, all_pred_flat))
    print(" ➤ MAE global :", mean_absolute_error(all_true_flat, all_pred_flat))
    print(" ➤ RMSE global:", np.sqrt(mean_squared_error(all_true_flat, all_pred_flat)))
else:
    print("\n⚠️ Pas de métriques globales (aucune date valide).")

## 10. PRÉDICTION CARTOGRAPHIQUE (FORECAST dates) + sauvegarde rasters

In [ ]:
# ------------------------------------------------------------
# 10) PRÉDICTION CARTOGRAPHIQUE (FORECAST dates) + sauvegarde rasters
# ------------------------------------------------------------
print("\n=== PRÉDICTION CARTOGRAPHIQUE (FORECAST) ===")

if template_profile is None:
    raise RuntimeError("Template GLDAS introuvable (aucun GWS lu).")

out_profile = template_profile.copy()
out_profile.update(dtype="float32", count=1, nodata=NODATA_VALUE)

all_forecast_dates = sorted(
    set(chirps_dict.keys())
    & set(lst_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

if not all_forecast_dates:
    raise RuntimeError("⚠️ Aucune date commune pour forecast (CHIRPS ∩ LST ∩ Qsb ∩ RZSM ∩ SoilM).")

user_start = FORECAST_START.date()
user_end   = FORECAST_END.date()
forecast_start_date = max(user_start, min(all_forecast_dates))
forecast_end_date   = min(user_end,   max(all_forecast_dates))

forecast_dates = [
    d for d in all_forecast_dates
    if (forecast_start_date <= d <= forecast_end_date) and (d not in EXCLUDED_DATES)
]

print("Fenêtre demandée :", user_start, "→", user_end)
print("Fenêtre utilisée :", forecast_start_date, "→", forecast_end_date)
print("Nb dates forecast :", len(forecast_dates))

region_mask = get_region_mask(template_profile)

for date_key in forecast_dates:
    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(f"⏭️ {date_key} : NDVI/ET indisponible")
        continue

    precip_raw, precip_prof, precip_nodata = read_raster_raw(chirps_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pix = precip_f.size
    X_all = np.stack([
        precip_f.ravel(),
        lst_f.ravel(),
        ndvi_f.ravel(),
        et_f.ravel(),
        qsb_f.ravel(),
        rzsm_f.ravel(),
        soilm_f.ravel(),
        np.full(n_pix, sin_doy, dtype="float32"),
        np.full(n_pix, cos_doy, dtype="float32"),
    ], axis=1).astype("float32")

    if np.isnan(X_all).any():
        print(f"⏭️ {date_key} : X_all contient NaN → ignorée")
        continue

    y_pred_all = knn_reg.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    out_name = os.path.join(OUT_DIR, f"GWS_PRED_KNN_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name, "w", **out_profile) as dst:
        dst.write(pred_clip, 1)

    print("✔️ Forecast sauvegardé:", out_name)

tif_files = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".tif")]
print(f"\n📂 Total rasters générés dans {OUT_DIR} : {len(tif_files)}")

## Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers présents dans le dossier de sortie, par extension
from collections import Counter

_count = Counter(os.path.splitext(_f)[1].lower() or "(sans extension)" for _f in os.listdir(OUT_DIR))
for _ext, _n in sorted(_count.items()):
    print(f"{_n:6d} fichier(s) {_ext}")
print("Dossier de sortie :", OUT_DIR)